# AFS Validate

Scans extracted financial data for anomalies:
- **YoY swing** — amount changes >10% between consecutive fiscal years
- **Digit mismatch** — digit count differs by ≥2 (strong OCR/LLM misread signal)

Flags are written to `COMMON.REVIEW_QUEUE` for analyst triage.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
import sys, pandas as pd
sys.path.insert(0, '../src')

from afs.snowflake_io import cursor_from_session
from afs.validate import validate_org, write_flags_to_review_queue, summarize_flags

In [ ]:
%%sql -r orgs
SELECT ORG_ID, ORG_CODE, LEGAL_NAME
  FROM AUDITED_FINANCIALS.COMMON.ORG_REGISTRY
 ORDER BY LEGAL_NAME

In [ ]:
ORG_CODE = orgs['ORG_CODE'].iloc[0]
org_id = orgs.loc[orgs['ORG_CODE'] == ORG_CODE, 'ORG_ID'].iloc[0]
print(f'Validating: {ORG_CODE} ({org_id})')

In [ ]:
with cursor_from_session(session, commit_on_exit=False) as cur:
    flags = validate_org(cur, org_id)

print(summarize_flags(flags))

In [ ]:
df_flags = pd.DataFrame(flags)
if not df_flags.empty:
    df_flags = df_flags.sort_values(['reason', 'statement', 'native_label'])
    display_cols = ['reason', 'statement', 'native_label', 'fy_label', 'amount', 'detail']
    print(df_flags[display_cols].to_string(index=False))
else:
    print('No flags found.')

In [ ]:
digit_flags = [f for f in flags if f['reason'] == 'digit_mismatch']
if digit_flags:
    print(f'{len(digit_flags)} CRITICAL digit-mismatch flag(s) — likely OCR/LLM misread:')
    for f in digit_flags:
        print(f"  {f['statement']}.{f['native_label']}  {f['detail']}")
else:
    print('No digit-mismatch flags.')

In [ ]:
WRITE_TO_REVIEW_QUEUE = False

if WRITE_TO_REVIEW_QUEUE and flags:
    with cursor_from_session(session) as cur:
        n = write_flags_to_review_queue(cur, org_id, flags)
    print(f'Wrote {n} flag(s) to COMMON.REVIEW_QUEUE.')
else:
    print(f'{len(flags)} flag(s) ready. Set WRITE_TO_REVIEW_QUEUE = True to persist.')

---
## Crossfoot Validation — Baptist Memorial FY2025

Verifies that extracted line-item details sum to their reported subtotals within each statement, and that key figures tie across IS / BS / CF.

In [ ]:
%%sql -r is_crossfoot
WITH fy25 AS (
    SELECT * FROM AUDITED_FINANCIALS.BAPTIST_MEMORIAL.IS_NATIVE
    WHERE FY_LABEL = 'FY2025'
)
SELECT 
    'Total Revenue' AS CHECK_NAME,
    (SELECT SUM(AMOUNT) FROM fy25 WHERE LINE_ORDER IN (1, 2) AND IS_SUBTOTAL = FALSE) AS COMPUTED,
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 3 AND IS_SUBTOTAL = TRUE) AS REPORTED,
    (SELECT SUM(AMOUNT) FROM fy25 WHERE LINE_ORDER IN (1, 2) AND IS_SUBTOTAL = FALSE) -
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 3 AND IS_SUBTOTAL = TRUE) AS DIFF
UNION ALL
SELECT 'Total Expenses',
    (SELECT SUM(AMOUNT) FROM fy25 WHERE LINE_ORDER BETWEEN 4 AND 10 AND IS_SUBTOTAL = FALSE),
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 11 AND IS_SUBTOTAL = TRUE),
    (SELECT SUM(AMOUNT) FROM fy25 WHERE LINE_ORDER BETWEEN 4 AND 10 AND IS_SUBTOTAL = FALSE) -
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 11 AND IS_SUBTOTAL = TRUE)
UNION ALL
SELECT 'Income from Operations (Rev - Exp)',
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 3) - (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 11),
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 12 AND IS_SUBTOTAL = TRUE),
    ((SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 3) - (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 11)) -
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 12 AND IS_SUBTOTAL = TRUE)
UNION ALL
SELECT 'Total Nonoperating Income',
    (SELECT SUM(AMOUNT) FROM fy25 WHERE LINE_ORDER BETWEEN 13 AND 20 AND IS_SUBTOTAL = FALSE),
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 21 AND IS_SUBTOTAL = TRUE),
    (SELECT SUM(AMOUNT) FROM fy25 WHERE LINE_ORDER BETWEEN 13 AND 20 AND IS_SUBTOTAL = FALSE) -
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 21 AND IS_SUBTOTAL = TRUE)
UNION ALL
SELECT 'Revenue in Excess of Expenses',
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 12) + (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 21),
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 22 AND IS_SUBTOTAL = TRUE),
    ((SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 12) + (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 21)) -
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 22 AND IS_SUBTOTAL = TRUE)
UNION ALL
SELECT 'Change in Net Assets (w/o + w/)',
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 27) + (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 32),
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 33 AND IS_SUBTOTAL = TRUE),
    ((SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 27) + (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 32)) -
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 33 AND IS_SUBTOTAL = TRUE)
UNION ALL
SELECT 'Ending Net Assets (Beg + Change)',
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 34) + (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 33),
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 35),
    ((SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 34) + (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 33)) -
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 35)

In [ ]:
%%sql -r bs_crossfoot
WITH fy25 AS (
    SELECT * FROM AUDITED_FINANCIALS.BAPTIST_MEMORIAL.BS_NATIVE
    WHERE FY_LABEL = 'FY2025'
)
SELECT 
    'Total Current Assets' AS CHECK_NAME,
    (SELECT SUM(AMOUNT) FROM fy25 WHERE LINE_ORDER BETWEEN 1 AND 8 AND IS_SUBTOTAL = FALSE) AS COMPUTED,
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 9 AND IS_SUBTOTAL = TRUE) AS REPORTED,
    (SELECT SUM(AMOUNT) FROM fy25 WHERE LINE_ORDER BETWEEN 1 AND 8 AND IS_SUBTOTAL = FALSE) -
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 9 AND IS_SUBTOTAL = TRUE) AS DIFF
UNION ALL
SELECT 'Total Assets',
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 9) +
    (SELECT SUM(COALESCE(AMOUNT,0)) FROM fy25 WHERE LINE_ORDER BETWEEN 10 AND 17 AND IS_SUBTOTAL = FALSE),
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 18 AND IS_SUBTOTAL = TRUE),
    ((SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 9) +
     (SELECT SUM(COALESCE(AMOUNT,0)) FROM fy25 WHERE LINE_ORDER BETWEEN 10 AND 17 AND IS_SUBTOTAL = FALSE)) -
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 18 AND IS_SUBTOTAL = TRUE)
UNION ALL
SELECT 'Total Current Liabilities',
    (SELECT SUM(AMOUNT) FROM fy25 WHERE LINE_ORDER BETWEEN 19 AND 24 AND IS_SUBTOTAL = FALSE),
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 25 AND IS_SUBTOTAL = TRUE),
    (SELECT SUM(AMOUNT) FROM fy25 WHERE LINE_ORDER BETWEEN 19 AND 24 AND IS_SUBTOTAL = FALSE) -
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 25 AND IS_SUBTOTAL = TRUE)
UNION ALL
SELECT 'Total Net Assets',
    (SELECT SUM(AMOUNT) FROM fy25 WHERE LINE_ORDER IN (31, 32) AND IS_SUBTOTAL = FALSE),
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 33 AND IS_SUBTOTAL = TRUE),
    (SELECT SUM(AMOUNT) FROM fy25 WHERE LINE_ORDER IN (31, 32) AND IS_SUBTOTAL = FALSE) -
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 33 AND IS_SUBTOTAL = TRUE)
UNION ALL
SELECT 'Total L+NA = Total Assets',
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 25) +
    (SELECT SUM(COALESCE(AMOUNT,0)) FROM fy25 WHERE LINE_ORDER BETWEEN 26 AND 29 AND IS_SUBTOTAL = FALSE) +
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 33),
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 34 AND IS_SUBTOTAL = TRUE),
    ((SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 25) +
     (SELECT SUM(COALESCE(AMOUNT,0)) FROM fy25 WHERE LINE_ORDER BETWEEN 26 AND 29 AND IS_SUBTOTAL = FALSE) +
     (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 33)) -
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 34 AND IS_SUBTOTAL = TRUE)
UNION ALL
SELECT 'Assets = L + NA (Balance Check)',
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 18),
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 34),
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 18) - (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 34)

In [ ]:
%%sql -r cf_crossfoot
WITH fy25 AS (
    SELECT * FROM AUDITED_FINANCIALS.BAPTIST_MEMORIAL.CF_NATIVE
    WHERE FY_LABEL = 'FY2025'
)
SELECT
    'Operating Activities' AS CHECK_NAME,
    (SELECT SUM(COALESCE(AMOUNT,0)) FROM fy25 WHERE SECTION = 'operating' AND IS_SUBTOTAL = FALSE) AS COMPUTED,
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 19 AND IS_SUBTOTAL = TRUE) AS REPORTED,
    (SELECT SUM(COALESCE(AMOUNT,0)) FROM fy25 WHERE SECTION = 'operating' AND IS_SUBTOTAL = FALSE) -
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 19 AND IS_SUBTOTAL = TRUE) AS DIFF
UNION ALL
SELECT 'Investing Activities',
    (SELECT SUM(COALESCE(AMOUNT,0)) FROM fy25 WHERE SECTION = 'investing' AND IS_SUBTOTAL = FALSE),
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 30 AND IS_SUBTOTAL = TRUE),
    (SELECT SUM(COALESCE(AMOUNT,0)) FROM fy25 WHERE SECTION = 'investing' AND IS_SUBTOTAL = FALSE) -
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 30 AND IS_SUBTOTAL = TRUE)
UNION ALL
SELECT 'Financing Activities',
    (SELECT SUM(COALESCE(AMOUNT,0)) FROM fy25 WHERE SECTION = 'financing' AND IS_SUBTOTAL = FALSE),
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 36 AND IS_SUBTOTAL = TRUE),
    (SELECT SUM(COALESCE(AMOUNT,0)) FROM fy25 WHERE SECTION = 'financing' AND IS_SUBTOTAL = FALSE) -
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 36 AND IS_SUBTOTAL = TRUE)
UNION ALL
SELECT 'Net Change = Oper + Invest + Fin',
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 19) + (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 30) + (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 36),
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 37),
    ((SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 19) + (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 30) + (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 36)) -
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 37)
UNION ALL
SELECT 'Ending Cash = Beg + Net Change',
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 38) + (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 37),
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 39),
    ((SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 38) + (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 37)) -
    (SELECT AMOUNT FROM fy25 WHERE LINE_ORDER = 39)

In [ ]:
%%sql -r cross_stmt
SELECT 
    'BS Cash = CF Ending Cash' AS CHECK_NAME,
    bs.AMOUNT AS STMT_A,
    cf.AMOUNT AS STMT_B,
    bs.AMOUNT - cf.AMOUNT AS DIFF
FROM AUDITED_FINANCIALS.BAPTIST_MEMORIAL.BS_NATIVE bs,
     AUDITED_FINANCIALS.BAPTIST_MEMORIAL.CF_NATIVE cf
WHERE bs.FY_LABEL = 'FY2025' AND bs.LINE_ORDER = 1 AND bs.IS_SUBTOTAL = FALSE
  AND cf.FY_LABEL = 'FY2025' AND cf.NATIVE_LABEL ILIKE '%end of year%'

UNION ALL

SELECT 'IS Depreciation = CF Depreciation',
    i.AMOUNT, cf.AMOUNT, i.AMOUNT - cf.AMOUNT
FROM AUDITED_FINANCIALS.BAPTIST_MEMORIAL.IS_NATIVE i,
     AUDITED_FINANCIALS.BAPTIST_MEMORIAL.CF_NATIVE cf
WHERE i.FY_LABEL = 'FY2025' AND i.NATIVE_LABEL ILIKE '%depreciation%' AND i.IS_SUBTOTAL = FALSE
  AND cf.FY_LABEL = 'FY2025' AND cf.NATIVE_LABEL ILIKE '%depreciation%' AND cf.IS_SUBTOTAL = FALSE

UNION ALL

SELECT 'BS Net Assets = IS Ending Net Assets',
    bs.AMOUNT, i.AMOUNT, bs.AMOUNT - i.AMOUNT
FROM AUDITED_FINANCIALS.BAPTIST_MEMORIAL.BS_NATIVE bs,
     AUDITED_FINANCIALS.BAPTIST_MEMORIAL.IS_NATIVE i
WHERE bs.FY_LABEL = 'FY2025' AND bs.NATIVE_LABEL = 'Total net assets' AND bs.IS_SUBTOTAL = TRUE
  AND i.FY_LABEL = 'FY2025' AND i.NATIVE_LABEL ILIKE 'NET ASSETS—End of year%'

UNION ALL

SELECT 'IS Change in NA = CF Change in NA',
    i.AMOUNT, cf.AMOUNT, i.AMOUNT - cf.AMOUNT
FROM AUDITED_FINANCIALS.BAPTIST_MEMORIAL.IS_NATIVE i,
     AUDITED_FINANCIALS.BAPTIST_MEMORIAL.CF_NATIVE cf
WHERE i.FY_LABEL = 'FY2025' AND i.NATIVE_LABEL = 'CHANGE IN NET ASSETS' AND i.IS_SUBTOTAL = TRUE
  AND cf.FY_LABEL = 'FY2025' AND cf.LINE_ORDER = 1

UNION ALL

SELECT 'CF Beg Cash = Prior FY BS Cash',
    cf.AMOUNT, bs.AMOUNT, cf.AMOUNT - bs.AMOUNT
FROM AUDITED_FINANCIALS.BAPTIST_MEMORIAL.CF_NATIVE cf,
     AUDITED_FINANCIALS.BAPTIST_MEMORIAL.BS_NATIVE bs
WHERE cf.FY_LABEL = 'FY2025' AND cf.NATIVE_LABEL ILIKE '%beginning%'
  AND bs.FY_LABEL = 'FY2024' AND bs.LINE_ORDER = 1 AND bs.IS_SUBTOTAL = FALSE
  AND bs.FILING_ID = 'ae4abd5d74d5ae29de4baa70dd3e63110ed6eaba48d193d775657102d37d8c90'

In [ ]:
import pandas as pd

all_checks = pd.concat([is_crossfoot, bs_crossfoot, cf_crossfoot, cross_stmt], ignore_index=True)
failed = all_checks[all_checks['DIFF'] != 0]
print(f'=== VALIDATION SUMMARY ===')
print(f'Total checks: {len(all_checks)}')
print(f'Passed:       {len(all_checks) - len(failed)}')
print(f'Failed:       {len(failed)}')
if not failed.empty:
    print('\nFAILED CHECKS:')
    print(failed.to_string(index=False))
else:
    print('\nAll crossfoot checks PASS.')